<a href="https://colab.research.google.com/github/sfreagin/Quantum-Programming/blob/main/Coding_qubit_gates_in_numpy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Programming single-qubit gates for a quantum computer

This notebook shows you how to create matrices and vectors for common quantum gates so you can test out the linear algebra operations

## There are two versions of each gate:
1. A version where you input individual qubits, i.e. separable (unentangled) states. These qubits must be 2D numpy column arrays:
  - e.g. `qubit = np.array([a,b]).reshape(-1,1)`
2. A version where you input the full state vector, which may include entangled state vectors. These must be a normal Python list:
  - e.g. `state = [a,b]`
  - **please note that the functions will automatically normalize the list vector in this second version** $\vec{x} \rightarrow \widehat{x} = \frac{\vec{x}}{|\vec{x}|} $

I also created several qubit states to check that the version (1) gates operate on an arbitrary qubit as described:
* The $|0>$ and $|1>$ states
* A test qubit $\psi = \frac{1}{\sqrt{3}}\begin{bmatrix} i \\ 2 \end{bmatrix}$

In [ ]:
import numpy as np

zero_state = np.array([1,0]).reshape(-1,1)
one_state = np.array([0,1]).reshape(-1,1)
test_state = np.array([1j/np.sqrt(3), np.sqrt(2) / np.sqrt(3)]).reshape(-1,1)

print(f"My test qubit is described by this numPy array:: \n{np.round(test_state,3)[:,0]}")
print(f"\nIt is normalized: the inner product with itself = {np.dot(np.conj(test_state.T), test_state)[:,0]}")

My test qubit is described by this numPy array:: 
[0.   +0.577j 0.816+0.j   ]

It is normalized: the inner product with itself = [1.+0.j]


In [ ]:
# I will loop through these objects in the example gates below
qubit_dict = {'|0>': zero_state,
              '|1>': one_state,
              '|test>': test_state}

angle_list = [0, 30, 45, 60, 90]

# I. NOT gate

The NOT gate takes qubit $\psi = \begin{bmatrix} a \\ b \end{bmatrix}$ and transforms it into $\psi' = \begin{bmatrix} b \\ a \end{bmatrix}$

$$X\psi = \begin{bmatrix} 0 & 1 \\ 1 & 0 \end{bmatrix}
\begin{bmatrix} a \\ b \end{bmatrix} = \begin{bmatrix} b \\ a \end{bmatrix}$$

In [ ]:
def not_gate(qubit):
  X = np.array([[0,1],[1,0]])
  return np.matmul(X, qubit)[:,0]

def not_gate_statevector(statevector):
  X = np.array([[0,1],[1,0]])
  statevector = np.array(statevector) / np.linalg.norm(statevector)
  return np.round(np.matmul(X, statevector),3)

### 1. NOT gate applied to several test qubits

In [ ]:
for i in qubit_dict.keys():
  print(f"\nNOT applied to the {i} state: \n{np.round(not_gate(qubit_dict[i]),3)}")


NOT applied to the |0> state: 
[0 1]

NOT applied to the |1> state: 
[1 0]

NOT applied to the |test> state: 
[0.816+0.j    0.   +0.577j]


### 2. NOT gate applied to a statevector list:

In [ ]:
not_gate_statevector([0.707j , 0.707])

array([0.707+0.j   , 0.   +0.707j])

# II. XOR (CNOT) gate

This gate takes two qubits as input and returns a conditional-NOT gate, i.e. a NOT gate is applied to the 2nd (target) qubit iff the 1st (control) qubit is a 1, and nothing happens if the control but is 0.

$$XOR|a,b> = |a, a \oplus b >$$

In the event that the control qubit is a superposition of 0 and 1, and XOR gates returns a superposition of outputs following the above rule.

$$XOR \psi =
\begin{bmatrix} 1 & 0 & 0 & 0. \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 0 & 1 \\ 0 & 0 & 1 & 0 \end{bmatrix}
\begin{bmatrix} a \\ b \\ c \\ d\end{bmatrix} =  \begin{bmatrix} a \\ b \\ d \\ c \end{bmatrix}$$

In [ ]:
def XOR_gate(qubit1, qubit2):
  XOR = np.array([[1,0,0,0],
                  [0,1,0,0],
                  [0,0,0,1],
                  [0,0,1,0]])
  full_state = np.kron(qubit1, qubit2)

  return np.matmul(XOR, full_state)[:,0]

def XOR_gate_statevector(statevector):
  XOR = np.array([[1,0,0,0],
                  [0,1,0,0],
                  [0,0,0,1],
                  [0,0,1,0]])
  statevector = np.array(statevector) / np.linalg.norm(statevector)

  return np.round(np.matmul(XOR, statevector),3)

### 1. XOR (CNOT) applied to multiple input combinations

In [ ]:
count = 0
for i in qubit_dict.keys():
  for j in qubit_dict.keys():
    count +=1
    print(f"\n{count}. Original state for {i} and {j} qubits:\n{np.round(np.kron(qubit_dict[i], qubit_dict[j]).reshape(1,4),3)[0]}")
    print(f"After sending {i} and {j} qubits through XOR")
    print(np.round(XOR_gate(qubit_dict[i], qubit_dict[j]),3).T)


1. Original state for |0> and |0> qubits:
[1 0 0 0]
After sending |0> and |0> qubits through XOR
[1 0 0 0]

2. Original state for |0> and |1> qubits:
[0 1 0 0]
After sending |0> and |1> qubits through XOR
[0 1 0 0]

3. Original state for |0> and |test> qubits:
[0.   +0.577j 0.816+0.j    0.   +0.j    0.   +0.j   ]
After sending |0> and |test> qubits through XOR
[0.   +0.577j 0.816+0.j    0.   +0.j    0.   +0.j   ]

4. Original state for |1> and |0> qubits:
[0 0 1 0]
After sending |1> and |0> qubits through XOR
[0 0 0 1]

5. Original state for |1> and |1> qubits:
[0 0 0 1]
After sending |1> and |1> qubits through XOR
[0 0 1 0]

6. Original state for |1> and |test> qubits:
[0.   +0.j    0.   +0.j    0.   +0.577j 0.816+0.j   ]
After sending |1> and |test> qubits through XOR
[0.   +0.j    0.   +0.j    0.816+0.j    0.   +0.577j]

7. Original state for |test> and |0> qubits:
[0.   +0.577j 0.   +0.j    0.816+0.j    0.   +0.j   ]
After sending |test> and |0> qubits through XOR
[0.   +0.577j 0.

### 2. XOR (CNOT) gate applied to a specified statevector list:

In [ ]:
print(XOR_gate_statevector([2,2,2,0]))

[0.577 0.577 0.    0.577]


# III. SWAP gate

This gate takes two qubits as input and swaps the states:

$$SWAP|a,b> = |b, a >$$


$$SWAP \psi =
\begin{bmatrix} 1 & 0 & 0 & 0 \\ 0 & 0 & 1 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 0 & 1 \end{bmatrix}
\begin{bmatrix} a \\ b \\ c \\ d\end{bmatrix} =  \begin{bmatrix} a \\ c \\ b \\ d \end{bmatrix}$$

In [ ]:
def SWAP_gate(qubit1, qubit2):
  SWAP = np.array([[1,0,0,0],
                   [0,0,1,0],
                   [0,1,0,0],
                   [0,0,0,1]])

  full_state = np.kron(qubit1, qubit2)
  return np.matmul(SWAP, full_state)[:,0]

def SWAP_gate_statevector(statevector):
  SWAP = np.array([[1,0,0,0],
                   [0,0,1,0],
                   [0,1,0,0],
                   [0,0,0,1]])

  statevector = np.array(statevector) / np.linalg.norm(statevector)
  return np.round(np.matmul(SWAP, statevector),3)

### 1. SWAP gate applied to multiple input combinations

In [ ]:
count = 0
for i in qubit_dict.keys():
  for j in qubit_dict.keys():
    count+=1
    print(f"\n{count}. Original state for {i} and {j} qubits:\n{np.round(np.kron(qubit_dict[i], qubit_dict[j]).reshape(1,4),3)[0]}")
    print(f"After sending {i} and {j} qubits through SWAP")
    print(np.round(SWAP_gate(qubit_dict[i], qubit_dict[j]),3).T)


1. Original state for |0> and |0> qubits:
[1 0 0 0]
After sending |0> and |0> qubits through SWAP
[1 0 0 0]

2. Original state for |0> and |1> qubits:
[0 1 0 0]
After sending |0> and |1> qubits through SWAP
[0 0 1 0]

3. Original state for |0> and |test> qubits:
[0.   +0.577j 0.816+0.j    0.   +0.j    0.   +0.j   ]
After sending |0> and |test> qubits through SWAP
[0.   +0.577j 0.   +0.j    0.816+0.j    0.   +0.j   ]

4. Original state for |1> and |0> qubits:
[0 0 1 0]
After sending |1> and |0> qubits through SWAP
[0 1 0 0]

5. Original state for |1> and |1> qubits:
[0 0 0 1]
After sending |1> and |1> qubits through SWAP
[0 0 0 1]

6. Original state for |1> and |test> qubits:
[0.   +0.j    0.   +0.j    0.   +0.577j 0.816+0.j   ]
After sending |1> and |test> qubits through SWAP
[0.   +0.j    0.   +0.577j 0.   +0.j    0.816+0.j   ]

7. Original state for |test> and |0> qubits:
[0.   +0.577j 0.   +0.j    0.816+0.j    0.   +0.j   ]
After sending |test> and |0> qubits through SWAP
[0.   +0.

### 2. SWAP gate applied to a specified statevector list:

In [ ]:
print(SWAP_gate_statevector([1j,0,2,2]))

[0.   +0.333j 0.667+0.j    0.   +0.j    0.667+0.j   ]


# IV. 1-bit Phase Shift gate

This gate takes one qubit and performs a rotation on the |1> state:


$$PS \psi =
\begin{bmatrix} 1 & 0 \\ 0 & e^{i\phi} \end{bmatrix}
\begin{bmatrix} a \\ b \end{bmatrix} =  \begin{bmatrix} a \\ be^{i\phi} \end{bmatrix}$$

In [ ]:
def phase1_gate(qubit, phi):
  phi = np.radians(phi)
  PS1 = np.array([[1,0],[0,np.exp(phi*1j)]])
  return np.matmul(PS1, qubit)[:,0]

def phase1_gate_statevector(statevector, phi):
  phi = np.radians(phi)
  PS1 = np.array([[1,0],[0,np.exp(phi*1j)]])
  statevector = np.array(statevector) / np.linalg.norm(statevector)
  return np.matmul(PS1, statevector)

### 1. Phase shift gate applied to several qubits and using different angles (note: all angles are in degrees)

In [ ]:
count = 0
for i in qubit_dict.keys():
  for j in angle_list:
    count += 1
    print(f"\n{count}. Phase shift {j} degrees applied to the {i} state: \n{np.round(phase1_gate(qubit_dict[i], j),3)}")


1. Phase shift 0 degrees applied to the |0> state: 
[1.+0.j 0.+0.j]

2. Phase shift 30 degrees applied to the |0> state: 
[1.+0.j 0.+0.j]

3. Phase shift 45 degrees applied to the |0> state: 
[1.+0.j 0.+0.j]

4. Phase shift 60 degrees applied to the |0> state: 
[1.+0.j 0.+0.j]

5. Phase shift 90 degrees applied to the |0> state: 
[1.+0.j 0.+0.j]

6. Phase shift 0 degrees applied to the |1> state: 
[0.+0.j 1.+0.j]

7. Phase shift 30 degrees applied to the |1> state: 
[0.   +0.j  0.866+0.5j]

8. Phase shift 45 degrees applied to the |1> state: 
[0.   +0.j    0.707+0.707j]

9. Phase shift 60 degrees applied to the |1> state: 
[0. +0.j    0.5+0.866j]

10. Phase shift 90 degrees applied to the |1> state: 
[0.+0.j 0.+1.j]

11. Phase shift 0 degrees applied to the |test> state: 
[0.   +0.577j 0.816+0.j   ]

12. Phase shift 30 degrees applied to the |test> state: 
[0.   +0.577j 0.707+0.408j]

13. Phase shift 45 degrees applied to the |test> state: 
[0.   +0.577j 0.577+0.577j]

14. Phase shift

### 2. One-bit phase shift gate applied to a specified statevector list:

In [ ]:
print(np.round(phase1_gate_statevector([0.707j , 0.707], 30),3))

[0.   +0.707j 0.612+0.354j]


# V. Controlled Phase Shift (2-bit phase shift) gate

This gate takes two qubits as input and returns a conditional phase shift gate, i.e. a phase shift gate is applied to the 2nd (target) qubit iff the 1st (control) qubit is a 1, and nothing happens if the control but is 0.

$$CPS_{2}|a,b> = e^{i(a\cdot b)\phi}|a,b >$$

In the event that the control qubit is a superposition of 0 and 1, the CPS_{2} gate returns a superposition of outputs following the above rule.

$$CPS_{2} \psi =
\begin{bmatrix} 1 & 0 & 0 & 0. \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 1 & 0 \\ 0 & 0 & 0 & e^{i\phi} \end{bmatrix}
\begin{bmatrix} a \\ b \\ c \\ d\end{bmatrix} =  \begin{bmatrix} a \\ b \\ c \\ de^{i\phi} \end{bmatrix}$$

In [ ]:
def phase2_gate(qubit1, qubit2, phi):
  phi = np.radians(phi)
  CPS2 = np.array([[1,0,0,0],
                  [0,1,0,0],
                  [0,0,1,0],
                  [0,0,0,np.exp(phi*1j)]])
  full_state = np.kron(qubit1, qubit2)[:,0]

  return np.matmul(CPS2, full_state)

def phase2_gate_statevector(statevector, phi):
  phi = np.radians(phi)
  CPS2 = np.array([[1,0,0,0],
                  [0,1,0,0],
                  [0,0,1,0],
                  [0,0,0,np.exp(phi*1j)]])

  statevector = np.array(statevector) / np.linalg.norm(statevector)

  return np.matmul(CPS2, statevector)

### 1. Controlled phase shift gate applied to several qubit combinations

In [ ]:
count = 0
for i in qubit_dict.keys():
  for j in qubit_dict.keys():
    for k in angle_list:
      count += 1
      print(f"\n{count}. Original state for {i} and {j} qubits:\n{np.round(np.kron(qubit_dict[i], qubit_dict[j]).reshape(1,4),3)[0]}")
      print(f"After controlled phase shift of {k} degrees:")
      print(np.round(phase2_gate(qubit_dict[i], qubit_dict[j], k),3).T)



1. Original state for |0> and |0> qubits:
[1 0 0 0]
After controlled phase shift of 0 degrees:
[1.+0.j 0.+0.j 0.+0.j 0.+0.j]

2. Original state for |0> and |0> qubits:
[1 0 0 0]
After controlled phase shift of 30 degrees:
[1.+0.j 0.+0.j 0.+0.j 0.+0.j]

3. Original state for |0> and |0> qubits:
[1 0 0 0]
After controlled phase shift of 45 degrees:
[1.+0.j 0.+0.j 0.+0.j 0.+0.j]

4. Original state for |0> and |0> qubits:
[1 0 0 0]
After controlled phase shift of 60 degrees:
[1.+0.j 0.+0.j 0.+0.j 0.+0.j]

5. Original state for |0> and |0> qubits:
[1 0 0 0]
After controlled phase shift of 90 degrees:
[1.+0.j 0.+0.j 0.+0.j 0.+0.j]

6. Original state for |0> and |1> qubits:
[0 1 0 0]
After controlled phase shift of 0 degrees:
[0.+0.j 1.+0.j 0.+0.j 0.+0.j]

7. Original state for |0> and |1> qubits:
[0 1 0 0]
After controlled phase shift of 30 degrees:
[0.+0.j 1.+0.j 0.+0.j 0.+0.j]

8. Original state for |0> and |1> qubits:
[0 1 0 0]
After controlled phase shift of 45 degrees:
[0.+0.j 1.+0.j 0

### 2. Controlled phase shift gate applied to a specified statevector list:

In [ ]:
print(np.round(phase2_gate_statevector([0, 0.707, 0.707, 1], 30),3))

[0.   +0.j    0.5  +0.j    0.5  +0.j    0.612+0.354j]


# VI. Toffoli (CCNOT) gate

This gate takes three qubits as input and performs a controlled-controlled-NOT, meaning you perform a NOT gate on the 3rd qubit iff the 1st and 2nd qubits are both in state 1:

$$T|a,b,c> = |a, b, (a\cdot b)\oplus c>$$

In the event that either control qubit (or both) are in a superposition of 0 and 1, and Toffoli gate returns a superposition of outputs following the above rule.

$$T \psi =
\begin{bmatrix} 1 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 & 0 & 0 & 0 & 0 \\ 0 & 0 & 1 & 0 & 0 & 0 & 0 & 0 \\ 0 & 0 & 0 & 1 & 0 & 0 & 0 & 0 \\ 0 & 0 & 0 & 0 & 1 & 0 & 0 & 0 \\ 0 & 0 & 0 & 0 & 0 & 1 & 0 & 0 \\ 0 & 0 & 0 & 0 & 0 & 0 & 0 & 1 \\ 0 & 0 & 0 & 0 & 0 & 0 & 1 & 0 \end{bmatrix}
\begin{bmatrix} a \\ b \\ c \\ d \\ e \\ f \\ g \\ h \end{bmatrix} =  \begin{bmatrix} a \\ b \\ c \\ d \\ e \\ f \\ h \\ g \end{bmatrix}$$

In [ ]:
def Toffoli_gate(qubit1, qubit2, qubit3):
  Toffoli = np.array([[1,0,0,0,0,0,0,0],
                      [0,1,0,0,0,0,0,0],
                      [0,0,1,0,0,0,0,0],
                      [0,0,0,1,0,0,0,0],
                      [0,0,0,0,1,0,0,0],
                      [0,0,0,0,0,1,0,0],
                      [0,0,0,0,0,0,0,1],
                      [0,0,0,0,0,0,1,0]])
  full_state = np.kron(qubit1, np.kron(qubit2, qubit3))
  return np.matmul(Toffoli, full_state)[:,0]

def Toffoli_gate_statevector(statevector):
  Toffoli = np.array([[1,0,0,0,0,0,0,0],
                      [0,1,0,0,0,0,0,0],
                      [0,0,1,0,0,0,0,0],
                      [0,0,0,1,0,0,0,0],
                      [0,0,0,0,1,0,0,0],
                      [0,0,0,0,0,1,0,0],
                      [0,0,0,0,0,0,0,1],
                      [0,0,0,0,0,0,1,0]])

  statevector = np.array(statevector) / np.linalg.norm(statevector)
  return np.matmul(Toffoli, statevector)

### 1. Toffoli gate applied to multiple input combinations

In [ ]:
count = 0
for i in qubit_dict.keys():
  for j in qubit_dict.keys():
    for k in qubit_dict.keys():
      count += 1
      print(f"\n{count}. Original state for {i}, {j}, and {k} qubits:")
      print(np.round(np.kron(qubit_dict[i], np.kron(qubit_dict[j], qubit_dict[k])).reshape(1,8),3)[0])
      print(f"After sending {i}, {j}, and {k} qubits through Toffoli gate:")
      print(np.round(Toffoli_gate(qubit_dict[i], qubit_dict[j], qubit_dict[k]).T, 3))


1. Original state for |0>, |0>, and |0> qubits:
[1 0 0 0 0 0 0 0]
After sending |0>, |0>, and |0> qubits through Toffoli gate:
[1 0 0 0 0 0 0 0]

2. Original state for |0>, |0>, and |1> qubits:
[0 1 0 0 0 0 0 0]
After sending |0>, |0>, and |1> qubits through Toffoli gate:
[0 1 0 0 0 0 0 0]

3. Original state for |0>, |0>, and |test> qubits:
[0.   +0.577j 0.816+0.j    0.   +0.j    0.   +0.j    0.   +0.j
 0.   +0.j    0.   +0.j    0.   +0.j   ]
After sending |0>, |0>, and |test> qubits through Toffoli gate:
[0.   +0.577j 0.816+0.j    0.   +0.j    0.   +0.j    0.   +0.j
 0.   +0.j    0.   +0.j    0.   +0.j   ]

4. Original state for |0>, |1>, and |0> qubits:
[0 0 1 0 0 0 0 0]
After sending |0>, |1>, and |0> qubits through Toffoli gate:
[0 0 1 0 0 0 0 0]

5. Original state for |0>, |1>, and |1> qubits:
[0 0 0 1 0 0 0 0]
After sending |0>, |1>, and |1> qubits through Toffoli gate:
[0 0 0 1 0 0 0 0]

6. Original state for |0>, |1>, and |test> qubits:
[0.   +0.j    0.   +0.j    0.   +0.577j 

### 2. Toffoli gate applied to a specified statevector list:

In [ ]:
print(np.round(Toffoli_gate_statevector([1,2,3,4,5,6,7,8]),3))

[0.07 0.14 0.21 0.28 0.35 0.42 0.56 0.49]


# VII. Five-bit Hadamard gate

I'm choosing to implement this as the tensor product of 5 Hadamard gates, i.e. $H^{\otimes 5} = H \otimes H \otimes H \otimes H \otimes H$ because it is easier for me to visualize compared with the equivalent Walsh-Hadamard sum:

$$H_{5} |y> = \frac{1}{2^{5/2}} \sum_{x=0}^{2^{5}-1}(-1)^{x\cdot y}|x>$$



In [ ]:
def five_hadamard_gate(qubit1, qubit2, qubit3, qubit4, qubit5):
  H = np.array([[1,1],[1,-1]]) / np.sqrt(2)
  full_H_gate = np.kron(H, np.kron(H, np.kron(H, np.kron(H,H))))

  full_input_state = np.kron(qubit1, np.kron(qubit2, np.kron(qubit3, np.kron(qubit4,qubit5))))

  return np.round(np.matmul(full_H_gate, full_input_state),3)[:,0]

def five_hadamard_gate_statevector(statevector):
  H = np.array([[1,1],[1,-1]]) / np.sqrt(2)
  full_H_gate = np.kron(H, np.kron(H, np.kron(H, np.kron(H,H))))

  statevector = np.array(statevector) / np.linalg.norm(statevector)

  return np.matmul(full_H_gate, statevector)

### 1. $H^{\otimes5}$ gate applied to a few common inputs

In [ ]:
# input: |00000>
print(five_hadamard_gate(zero_state,zero_state,zero_state, zero_state,zero_state).T)

[0.177 0.177 0.177 0.177 0.177 0.177 0.177 0.177 0.177 0.177 0.177 0.177
 0.177 0.177 0.177 0.177 0.177 0.177 0.177 0.177 0.177 0.177 0.177 0.177
 0.177 0.177 0.177 0.177 0.177 0.177 0.177 0.177]


In [ ]:
# input: |11111>
print(five_hadamard_gate(one_state, one_state, one_state, one_state, one_state).T)

[ 0.177 -0.177 -0.177  0.177 -0.177  0.177  0.177 -0.177 -0.177  0.177
  0.177 -0.177  0.177 -0.177 -0.177  0.177 -0.177  0.177  0.177 -0.177
  0.177 -0.177 -0.177  0.177  0.177 -0.177 -0.177  0.177 -0.177  0.177
  0.177 -0.177]


In [ ]:
# input: |10101>
print(five_hadamard_gate(one_state, zero_state, one_state, zero_state, one_state).T)

[ 0.177 -0.177  0.177 -0.177 -0.177  0.177 -0.177  0.177  0.177 -0.177
  0.177 -0.177 -0.177  0.177 -0.177  0.177 -0.177  0.177 -0.177  0.177
  0.177 -0.177  0.177 -0.177 -0.177  0.177 -0.177  0.177  0.177 -0.177
  0.177 -0.177]


### 2. $H^{\otimes5}$ gate applied to a specified statevector list:

In [ ]:
print(np.round(five_hadamard_gate_statevector([1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32]),3))

[ 0.873 -0.026 -0.053  0.    -0.106  0.     0.    -0.    -0.212  0.
  0.    -0.    -0.    -0.    -0.     0.    -0.423  0.     0.    -0.
 -0.    -0.     0.    -0.    -0.     0.    -0.     0.     0.     0.
  0.    -0.   ]
